In [ ]:
# DATASET PREPARATION

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
from PIL import Image
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.models.segmentation as models
import torch.nn as nn
import torch
import matplotlib.pyplot as plt

In [ ]:
class RoadBaseDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_names = sorted(os.listdir(images_dir))
        self.mask_names = sorted(os.listdir(masks_dir))

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        mask_name = self.mask_names[idx]
        img_path = os.path.join(self.images_dir, image_name)
        mask_path = os.path.join(self.masks_dir, mask_name)

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path).convert("L"))
        return image, mask

In [ ]:
IMG_SIZE = 512

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(0.0625, 0.0625), rotate=(-45, 45), p=0.5, interpolation=1, mask_interpolation=0),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class IndexedRoadDataset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset      
        self.indices = indices                
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        image_name = self.base_dataset.image_names[idx]
        mask_name = self.base_dataset.mask_names[idx]

        img_path = os.path.join(self.base_dataset.images_dir, image_name)
        mask_path = os.path.join(self.base_dataset.masks_dir, mask_name)

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path).convert("L"))

        mask = mask.astype(np.int64)

        augmented = self.transform(image=image, mask=mask)
        image = augmented["image"].float()
        
        mask  = augmented["mask"].long()

        return image, mask

In [ ]:
images_dir = "https://drive.google.com/drive/folders/1IMeO1J9DjUaXONHnT84ThJBS2rkVMfVZ?usp=sharing"
masks_dir  = "https://drive.google.com/drive/folders/1rytAA4Ipfx0Ia-K4pzchZXE-kODDKXHq?usp=sharing"

base_dataset = RoadBaseDataset(images_dir, masks_dir)

print("len(base_dataset):", len(base_dataset))

img0, m0 = base_dataset[0]
print("base_dataset[0] image:", type(img0), img0.shape)
print("base_dataset[0] mask: ", type(m0),  m0.shape)

n = len(base_dataset) 
n_train = int(0.7 * n)
n_val = int(0.20 * n)
n_test = int(0.1 * n)

indices = list(range(n))

print("n_train: ", n_train)
print("n_val: ", n_val)
print("n_test: ", n_test)

train_indices = indices[:n_train]
val_indices   = indices[n_train:(n_train+n_val)]
test_indices   = indices[(n_train+n_val):]

print("train_indices:", train_indices)
print("val_indices:  ", val_indices)
print("test_indices:  ", test_indices)

train_ds = IndexedRoadDataset(base_dataset, train_indices, train_transform)
val_ds   = IndexedRoadDataset(base_dataset, val_indices,   val_transform)
test_ds = IndexedRoadDataset(base_dataset, test_indices, val_transform)

print("len(train_ds):", len(train_ds))
print("len(val_ds):  ", len(val_ds))
print("len(test_ds):  ", len(test_ds))

img1, m1 = train_ds[0]
print("train_ds[0] image:", img1.shape)
print("train_ds[0] mask: ", m1.shape)

train_loader = DataLoader(
    train_ds, 
    batch_size=2, 
    shuffle=True,  
    num_workers=0
)
val_loader   = DataLoader(
    val_ds,   
    batch_size=2, 
    shuffle=False, 
    num_workers=0
)

In [ ]:
# VISUALIZATION OF DATASET

In [ ]:
def visualize(**images) -> None:

    n = len(images)
    plt.figure(figsize=(16, 5))
    for i, (name, image) in enumerate(images.items()):
        plt.subplot(1, n, i + 1)
        plt.xticks([]) 
        plt.yticks([]) 
        plt.title(" ".join(name.split("_")).title()) 
        plt.imshow(image, vmin=0, vmax=10)
    plt.show()

In [ ]:
class UnNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):

        for t, m, s in zip(tensor, self.mean, self.std):
            t.mul_(s).add_(m) 

        return tensor


unnorm = UnNormalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

In [ ]:
idx = np.random.randint(len(train_ds)) 
image, mask = train_ds[idx] 
visualize(image=unnorm(image).permute(1, 2, 0), mask=mask)

In [ ]:
# MODEL 

In [ ]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

def create_model():
    model = smp.Unet(
        encoder_name="resnet50",        
        encoder_weights="imagenet",     
        in_channels=3,                  
        classes=4                       
    )
    return model

model = create_model().to(device)

In [ ]:
# TRAIN

In [ ]:
from tqdm.notebook import tqdm
import torch

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0

    miou_metric.reset()
    acc_metric.reset()

    train_progress = tqdm(loader, colour="green")

    for images, masks in train_progress:
        images = images.to(device)
        masks = masks.to(device)

         
        outputs = model(images)

        loss = loss_fn(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)  

        miou_metric.update(preds, masks)
        acc_metric.update(preds, masks)

    epoch_loss = running_loss / len(loader.dataset)

    return (
        epoch_loss,
        miou_metric.compute().item(),
        acc_metric.compute().item(),
    )

In [ ]:
@torch.no_grad()
def eval_one_epoch(model, loader, loss_fn, device):
    model.eval()
    running_loss = 0.0

    miou_metric.reset()
    acc_metric.reset()

    miou_metric_per_class.reset()
    acc_metric_per_class.reset()

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = loss_fn(outputs, masks)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        miou_metric.update(preds, masks)
        acc_metric.update(preds, masks)

        miou_metric_per_class.update(preds, masks)
        acc_metric_per_class.update(preds, masks)

        per_class_iou = miou_metric_per_class.compute().cpu().tolist()
        per_class_acc = acc_metric_per_class.compute().cpu().tolist()

    epoch_loss = running_loss / len(loader.dataset)

    return (
        epoch_loss,
        miou_metric.compute().item(),
        acc_metric.compute().item(),

        per_class_iou,
        per_class_acc,
    )


In [ ]:
# HYPERPARAMETERS

In [ ]:
import torch.optim.lr_scheduler

cross_entropy_loss = nn.CrossEntropyLoss(ignore_index=255)

def loss_fn(outputs, masks):
    return cross_entropy_loss(outputs, masks)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [ ]:
## METRICS

In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassAccuracy
from torchmetrics.segmentation import DiceScore

NUM_CLASSES = 4
IGNORE_INDEX = 255

miou_metric = MulticlassJaccardIndex(
    num_classes=NUM_CLASSES,
    average="macro",
    ignore_index=IGNORE_INDEX
).to(device)

acc_metric = MulticlassAccuracy(
    num_classes=NUM_CLASSES,
    average="macro",
    ignore_index=IGNORE_INDEX
).to(device)

miou_metric_per_class = MulticlassJaccardIndex(
    num_classes=NUM_CLASSES,
    average=None,
    ignore_index=IGNORE_INDEX
).to(device)

acc_metric_per_class = MulticlassAccuracy(
    num_classes=NUM_CLASSES,
    average=None,
    ignore_index=IGNORE_INDEX
).to(device)


In [ ]:
import pandas as pd

metrics_history = {
    "epoch": [],
    "train_loss": [],
    "train_mIoU": [],
    "train_acc": [],
    "val_loss": [],
    "val_mIoU": [],
    "val_acc": [],
    "val_miou_class": [],
    "val_acc_class": [],
}


In [ ]:
## CHECKPOINTS

In [ ]:
checkpoints_dir = "checkpoints_patfinder"
os.makedirs(checkpoints_dir, exist_ok=True)

In [ ]:
def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    val_loss,
    path
):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "val_loss": val_loss,
    }

    torch.save(checkpoint, path)

In [ ]:
# TRAINING

In [ ]:
num_epochs = 40

best_val_loss = float("inf")

for epoch in range(num_epochs):
    train_loss, train_miou, train_acc = train_one_epoch(
        model, train_loader, optimizer, loss_fn, device
    )

    val_loss, val_miou, val_acc, val_miou_class, val_acc_class = eval_one_epoch(
        model, val_loader, loss_fn, device
    )

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch+1}/{num_epochs} | lr: {optimizer.param_groups[0]['lr']:.6f} \n"
        f"Train: loss={train_loss:.4f}, mIoU={train_miou:.4f}, Acc={train_acc:.4f} | \n"
        f"Val: loss={val_loss:.4f}, mIoU={val_miou:.4f}, Acc={val_acc:.4f} \n"
    )


    save_checkpoint(
        epoch=epoch + 1,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        val_loss=val_loss,
        path=os.path.join(checkpoints_dir, "checkpoints_last.pt")
    )

  
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            epoch=epoch + 1,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            val_loss=val_loss,
            path=os.path.join(checkpoints_dir, "checkpoints_best.pt")
        )
        print("Best model saved")

    metrics_history["epoch"].append(epoch + 1)

    metrics_history["train_loss"].append(train_loss)
    metrics_history["train_mIoU"].append(train_miou)
    metrics_history["train_acc"].append(train_acc)

    metrics_history["val_loss"].append(val_loss)
    metrics_history["val_mIoU"].append(val_miou)
    metrics_history["val_acc"].append(val_acc)


    metrics_history["val_miou_class"].append(val_miou_class)
    metrics_history["val_acc_class"].append(val_acc_class)


In [ ]:
df = pd.DataFrame(metrics_history)

df.to_csv("metrics.csv", index=False)
print("Metrics saved to training_metrics.csv")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


df = pd.read_csv(
    "/metrics.csv"
)


df_epoch = df.groupby("epoch").mean(numeric_only=True)


metrics = [
    ("train_loss", "val_loss", "Loss"),
    ("train_mIoU", "val_mIoU", "mIoU"),
]

for train_metric, val_metric, title in metrics:
    plt.figure(figsize=(8, 5))

    if train_metric in df_epoch.columns:
        plt.plot(
            df_epoch.index,
            df_epoch[train_metric],
            marker="o",
            label=train_metric
        )

    if val_metric in df_epoch.columns:
        plt.plot(
            df_epoch.index,
            df_epoch[val_metric],
            marker="o",
            label=val_metric
        )

    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
PRED_DIR = "/predictions_masks"
os.makedirs(PRED_DIR, exist_ok=True)

In [ ]:
# VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt
import cv2

@torch.no_grad()
def visualize_sample(
    model,
    dataset,
    idx=0,
    device=device,
    threshold=0.0, 
    save_dir=None
):


    ckpt_path = os.path.join(checkpoints_dir, "checkpoints_best.pt")

    checkpoint = torch.load(ckpt_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)

    model.eval()
    image, mask_true = dataset[idx]
    image_batch = image.unsqueeze(0).to(device)
    
    output = model(image_batch)["out"]

    softmax_output = torch.softmax(output, dim=1) # (1, 4, H, W)

    pred_classes = torch.argmax(softmax_output, dim=1) # (1, H, W)

    pred_probs, _ = torch.max(softmax_output, dim=1) # (1, H, W)

    pred_mask_np = pred_classes[0].cpu().numpy()
    pred_probs_np = pred_probs[0].cpu().numpy()

    pred_mask_np[pred_probs_np < threshold] = 0

    if save_dir is not None:
        base_name = dataset.base_dataset.image_names[dataset.indices[idx]]
        save_path = os.path.join(save_dir, base_name)

        cv2.imwrite(save_path, pred_mask_np.astype(np.uint8))

    img_np = image.permute(1,2,0).cpu().numpy()
    img_vis = img_np * 0.5 + 0.5       
    img_vis = np.clip(img_vis, 0, 1)

    true_np = mask_true.cpu().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(12,4))
    axs[0].imshow(img_vis)
    axs[0].set_title("Image")
    axs[1].imshow(true_np, cmap="viridis", vmin=0, vmax=3)
    axs[1].set_title("True mask")
    axs[2].imshow(pred_mask_np, cmap="viridis", vmin=0, vmax=3)
    axs[2].set_title(f"Pred mask (Threshold: {threshold:.2f})") 
    for ax in axs:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

visualization_threshold = 0.51 

for i in range(len(test_ds)):
    print(f"Sample {i}")
    visualize_sample(
        model,
        test_ds,
        idx=i,
        threshold=visualization_threshold,
        save_dir=PRED_DIR,
    )

print(f"Prediction masks saved to: {PRED_DIR}")

In [ ]:
# EVALUATION

In [ ]:
@torch.no_grad()
def evaluate_test(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0

    miou_metric.reset()
    acc_metric.reset()

    miou_metric_per_class.reset()
    acc_metric_per_class.reset()


    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = loss_fn(outputs, masks)
        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        miou_metric.update(preds, masks)
        acc_metric.update(preds, masks)

        miou_metric_per_class.update(preds, masks)
        acc_metric_per_class.update(preds, masks)

    test_loss = running_loss / len(loader.dataset)

    mean_iou = miou_metric.compute().item()
    mean_acc = acc_metric.compute().item()

    per_class_iou = miou_metric_per_class.compute().cpu().tolist()
    per_class_acc = acc_metric_per_class.compute().cpu().tolist()

    return (
        test_loss,
        mean_iou,
        mean_acc,
        per_class_iou,
        per_class_acc
    )

In [ ]:
import pandas as pd

test_loader   = DataLoader(
    test_ds,   
    batch_size=2, 
    shuffle=False, 
    num_workers=0
)

results = []

ckpt_path = os.path.join(checkpoints_dir, "checkpoints_best.pt")

experiments = [
    ckpt_path
]

for ckpt in experiments:

    checkpoint = torch.load(ckpt, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)

    test_loss, test_miou, test_acc, test_class_iou, test_class_acc = evaluate_test(
        model, test_loader, loss_fn, device
    )

    print(f"\nResults for {ckpt}")
    print("test_loss", test_loss)
    print("mIoU:", test_miou)
    print("Accuracy:", test_acc)

    results.append({
        "experiment": ckpt,
        "test_loss": test_loss,
        "test_mIoU": test_miou,
        "test_accuracy": test_acc, 
        "test_class_iou": test_class_iou,
        "test_class_acc": test_class_acc
    })

df = pd.DataFrame(results)

df.to_csv("test_results.csv", index=False)